# App Showcase: Breast Cancer Classification on Mammograms

BN4601A individual assignment — a standalone demo of a YOLO11s classifier that labels whole screening mammograms as **benign** or **malignant**.

Run all cells. Dependencies install automatically, and the trained weights and example images are pulled from the public Space [`Claeby/BN4601_indiv1`](https://huggingface.co/spaces/Claeby/BN4601_indiv1), so no additional files are needed. The final cell launches a Gradio interface and prints a temporary public URL.

Trained on CBIS-DDSM mass cases with a patient-level split. Held-out test performance (n=361): AUC 0.767, with sensitivity 0.786 and specificity 0.551 at the deployed referral threshold of 0.236.

**Educational demonstration only — not a medical device, and not for diagnosis or clinical decision-making.**

In [ ]:
!pip install -q ultralytics gradio huggingface_hub

In [ ]:
from pathlib import Path

import gradio as gr
from huggingface_hub import hf_hub_download
from ultralytics import YOLO

HF_REPO = 'Claeby/BN4601_indiv1'


def fetch(filename):
    local = Path('space_deploy') / filename
    if local.exists():
        return str(local)
    return hf_hub_download(repo_id=HF_REPO, filename=filename, repo_type='space')


print('✓ Libraries imported')

In [ ]:
# Referral threshold on P(malignant), tuned on the VALIDATION set for >=85% sensitivity
# and applied unchanged to the held-out test set (0.786 sensitivity / 0.551 specificity there).
# Kept in sync with space_deploy/app.py — change both together if you re-tune it.
REFERRAL_THRESHOLD = 0.236

MODEL_PATH = fetch('best.pt')

model = YOLO(MODEL_PATH)
idx_to_name = model.names
malignant_idx = [i for i, n in idx_to_name.items() if n == 'malignant'][0]

print('Loaded weights:', MODEL_PATH)
print('Classes:', idx_to_name)
print('Referral threshold:', REFERRAL_THRESHOLD)

In [ ]:
def classify_mammogram(image):
    if image is None:
        return {}, 'Upload a mammogram to see a result.'

    results = model.predict(source=image, verbose=False)
    probs = results[0].probs.data.cpu().numpy()
    scores = {idx_to_name[i]: float(probs[i]) for i in range(len(probs))}

    p_malignant = float(probs[malignant_idx])
    if p_malignant >= REFERRAL_THRESHOLD:
        decision = (
            f'### ⚠️ FLAG FOR REVIEW\n'
            f'P(malignant) = **{p_malignant:.1%}** — at or above the '
            f'{REFERRAL_THRESHOLD:.0%} referral threshold.'
        )
    else:
        decision = (
            f'### ✅ No flag raised\n'
            f'P(malignant) = **{p_malignant:.1%}** — below the '
            f'{REFERRAL_THRESHOLD:.0%} referral threshold.'
        )
    decision += (
        '\n\nThe threshold is deliberately set below 50% so the system errs toward '
        'over-referral, reflecting the asymmetric cost of errors in screening.'
    )
    return scores, decision

print('✓ classify_mammogram() ready')

In [ ]:
example_imgs = [fetch('example_benign.jpg'), fetch('example_malignant.jpg')]

demo = gr.Interface(
    fn=classify_mammogram,
    inputs=gr.Image(type='pil', label='Upload Screening Mammogram'),
    outputs=[
        gr.Label(num_top_classes=2, label='Class Probabilities'),
        gr.Markdown(label='Triage Decision'),
    ],
    title='AI Perception System: Breast Cancer Classification on Mammography',
    description=(
        'Upload a full screening mammogram to estimate **Benign** vs. **Malignant**.\n\n'
        'YOLO11s classifier trained on CBIS-DDSM mass cases with a patient-level split. '
        'Held-out test AUC 0.767 (n=361).\n\n'
        '**Educational demonstration only — not a medical device, and not for '
        'diagnosis or clinical decision-making.** (BN4601A)'
    ),
    examples=example_imgs,
)

# share=True gives a temporary public gradio.live URL (valid ~72 hours) so you can
# open it in a normal browser tab to screen-record, even without the HF Space working.
# If share=True fails on your network, set it to False and record the local window instead.
demo.launch(share=True)